# Amparo -- M2: Evaluacion del modelo fine-tuneado

Este notebook corre el harness de evaluacion de M2 sobre el adaptador LoRA
que entreno M1 (`baseline_finetune.ipynb`, ya publicado en la wiki). No
modifica ni depende de que M1 haya dejado archivos guardados: vuelve a
generar las respuestas baseline y fine-tuned sobre el **mismo split de
validacion** (mismo seed) para que los numeros sean comparables con el
baseline ya publicado (3.4% / 17.7% de similitud lexica).

Fases: generar respuestas (baseline y fine-tuned) -> liberar el adaptador y
reusar el modelo base como juez -> LLM-as-judge + sondeo de position bias ->
metricas clasicas + metrica de dominio juridico (sin GPU) -> sesgos de
longitud y auto-preferencia -> scorecard, persistido en Drive.

Toda la logica pesada vive en `tools/evaluation/` (repo principal, no en
este notebook) -- este notebook solo clona el repo, instala dependencias, y
orquesta las llamadas.

Antes de correr: `Entorno de ejecucion > Cambiar tipo de entorno de
ejecucion > GPU`.


In [1]:
# Clona el repo (o actualiza si ya existe de una corrida anterior en esta VM)
import os

if not os.path.isdir("Amparo"):
    !git clone https://github.com/TomasPosada0626/Amparo.git
else:
    !cd Amparo && git pull

%cd Amparo


Cloning into 'Amparo'...
remote: Enumerating objects: 87, done.
remote: Counting objects: 100% (87/87), done.
remote: Compressing objects: 100% (69/69), done.
remote: Total 87 (delta 24), reused 72 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (87/87), 223.67 KiB | 18.64 MiB/s, done.
Resolving deltas: 100% (24/24), done.
/content/Amparo


In [2]:
# Dependencias puras de tools/evaluation (metricas clasicas, tests)
!pip install -q -r requirements.txt


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 388.1/388.1 kB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 14.2 MB/s eta 0:00:00


In [3]:
# Stack de ML pesado -- igual que en M1 (baseline_finetune.ipynb, celda 1):
# se instala aqui y NO en requirements.txt del repo, para no arriesgar
# reemplazar el build de PyTorch con CUDA que Colab ya trae preinstalado.
# bert-score va aqui tambien (no en requirements.txt) porque depende de
# torch de forma transitiva.
!pip install -q -U transformers peft bitsandbytes accelerate bert-score
!pip install -U "bitsandbytes>=0.46.1"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.8 MB/s eta 0:00:00


## Configuracion

Las constantes (seed, val_fraction, modelo base, rutas de Drive) viven en
`tools/evaluation/config.py` -- deben coincidir con `RANDOM_SEED=42` y
`VAL_FRACTION=0.15` de M1 para evaluar sobre el mismo split.


In [4]:
from google.colab import drive

drive.mount('/content/drive')

from tools.evaluation import config, dataset

records = dataset.load_records()
train_records, val_records = dataset.stratified_split(records)
system_prompt = dataset.system_prompt(records)

print(f"Total: {len(records)} | train: {len(train_records)} | val: {len(val_records)}")
assert len(val_records) == 201, "El split no coincide con el de M1 -- revisar RANDOM_SEED/VAL_FRACTION"
print("Split verificado: coincide con el usado en M1.")


Mounted at /content/drive
Total: 1320 | train: 1119 | val: 201
Split verificado: coincide con el usado en M1.


## Fase 1 -- Generacion: baseline (modelo sin fine-tuning)

Puede tardar varios minutos segun el tamano de la validacion (201 ejemplos).


In [5]:
from tools.evaluation import generation

model, tokenizer = generation.load_base_model()

baseline_results = generation.generate_batch(
    model, tokenizer, system_prompt, val_records, label="baseline"
)
print(f"Generadas {len(baseline_results)} respuestas baseline.")


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

[baseline] 10/201 (5%) -- 11.4s/ejemplo, ETA ~36.2 min
[baseline] 20/201 (10%) -- 11.4s/ejemplo, ETA ~34.5 min
[baseline] 30/201 (15%) -- 12.2s/ejemplo, ETA ~34.6 min
[baseline] 40/201 (20%) -- 12.1s/ejemplo, ETA ~32.5 min
[baseline] 50/201 (25%) -- 11.7s/ejemplo, ETA ~29.4 min
[baseline] 60/201 (30%) -- 11.8s/ejemplo, ETA ~27.7 min
[baseline] 70/201 (35%) -- 11.7s/ejemplo, ETA ~25.6 min
[baseline] 80/201 (40%) -- 11.5s/ejemplo, ETA ~23.3 min
[baseline] 90/201 (45%) -- 11.6s/ejemplo, ETA ~21.5 min
[baseline] 100/201 (50%) -- 11.8s/ejemplo, ETA ~19.8 min
[baseline] 110/201 (55%) -- 11.8s/ejemplo, ETA ~17.9 min
[baseline] 120/201 (60%) -- 11.8s/ejemplo, ETA ~15.9 min
[baseline] 130/201 (65%) -- 11.7s/ejemplo, ETA ~13.9 min
[baseline] 140/201 (70%) -- 11.9s/ejemplo, ETA ~12.1 min
[baseline] 150/201 (75%) -- 12.0s/ejemplo, ETA ~10.2 min
[baseline] 160/201 (80%) -- 11.9s/ejemplo, ETA ~8.1 min
[baseline] 170/201 (85%) -- 11.9s/ejemplo, ETA ~6.2 min
[baseline] 180/201 (90%) -- 12.0s/ejemplo, 

## Fase 2 -- Generacion: fine-tuned (adaptador LoRA desde Drive)


In [6]:
model = generation.attach_adapter(model, config.DRIVE_ADAPTER_DIR)
model.eval()

finetuned_results = generation.generate_batch(
    model, tokenizer, system_prompt, val_records, label="fine_tuned"
)
print(f"Generadas {len(finetuned_results)} respuestas fine-tuned.")


[fine_tuned] 10/201 (5%) -- 4.5s/ejemplo, ETA ~14.2 min
[fine_tuned] 20/201 (10%) -- 4.4s/ejemplo, ETA ~13.2 min
[fine_tuned] 30/201 (15%) -- 4.3s/ejemplo, ETA ~12.4 min
[fine_tuned] 40/201 (20%) -- 4.4s/ejemplo, ETA ~11.7 min
[fine_tuned] 50/201 (25%) -- 4.3s/ejemplo, ETA ~10.9 min
[fine_tuned] 60/201 (30%) -- 4.3s/ejemplo, ETA ~10.1 min
[fine_tuned] 70/201 (35%) -- 4.3s/ejemplo, ETA ~9.3 min
[fine_tuned] 80/201 (40%) -- 4.3s/ejemplo, ETA ~8.6 min
[fine_tuned] 90/201 (45%) -- 4.2s/ejemplo, ETA ~7.9 min
[fine_tuned] 100/201 (50%) -- 4.2s/ejemplo, ETA ~7.1 min
[fine_tuned] 110/201 (55%) -- 4.2s/ejemplo, ETA ~6.4 min
[fine_tuned] 120/201 (60%) -- 4.3s/ejemplo, ETA ~5.8 min
[fine_tuned] 130/201 (65%) -- 4.2s/ejemplo, ETA ~5.0 min
[fine_tuned] 140/201 (70%) -- 4.2s/ejemplo, ETA ~4.3 min
[fine_tuned] 150/201 (75%) -- 4.3s/ejemplo, ETA ~3.6 min
[fine_tuned] 160/201 (80%) -- 4.3s/ejemplo, ETA ~2.9 min
[fine_tuned] 170/201 (85%) -- 4.2s/ejemplo, ETA ~2.2 min
[fine_tuned] 180/201 (90%) -- 4.2s/

## Fase 3 -- Liberar el adaptador y reutilizar el modelo base como juez

`detach_adapter` usa `model.unload()` (quita las capas LoRA sin fusionarlas)
-- así el mismo modelo cargado en memoria sirve como juez independiente del
adaptador, sin una segunda carga completa de modelo.


In [7]:
import torch

model = generation.detach_adapter(model)
torch.cuda.empty_cache()
print("Adaptador liberado. El modelo en memoria es ahora el juez (base, sin fine-tuning).")


Adaptador liberado. El modelo en memoria es ahora el juez (base, sin fine-tuning).


## Fase 4 -- LLM-as-a-Judge

Puntuacion absoluta (1-5 por criterio) de cada respuesta contra la
referencia del dataset, mas un sub-experimento de *position bias* sobre una
muestra de 30 pares baseline/fine-tuned.


In [8]:
from tools.evaluation import judge

judge_baseline = judge.score_batch(model, tokenizer, baseline_results)
judge_finetuned = judge.score_batch(model, tokenizer, finetuned_results)

n_fail_base = sum(1 for s in judge_baseline if not s.parse_ok)
n_fail_ft = sum(1 for s in judge_finetuned if not s.parse_ok)
print(f"Juez baseline: {len(judge_baseline)} filas, {n_fail_base} fallos de parseo.")
print(f"Juez fine-tuned: {len(judge_finetuned)} filas, {n_fail_ft} fallos de parseo.")


Juez baseline: 201 filas, 4 fallos de parseo.
Juez fine-tuned: 201 filas, 0 fallos de parseo.


In [9]:
from tools.evaluation import bias

pairs = [
    (b.id, b.query, b.generated, f.generated)
    for b, f in zip(baseline_results, finetuned_results)
]
position_bias_report = bias.run_position_bias_probe(model, tokenizer, pairs)
print(position_bias_report)


PositionBiasReport(n_pairs=30, n_flipped=0, n_tied_or_unparsed=0, flip_rate_pct=0.0, details=[{'id': 554, 'verdict_normal': 'baseline', 'verdict_swapped': 'baseline'}, {'id': 949, 'verdict_normal': 'baseline', 'verdict_swapped': 'baseline'}, {'id': 402, 'verdict_normal': 'baseline', 'verdict_swapped': 'baseline'}, {'id': 1061, 'verdict_normal': 'baseline', 'verdict_swapped': 'baseline'}, {'id': 886, 'verdict_normal': 'baseline', 'verdict_swapped': 'baseline'}, {'id': 1147, 'verdict_normal': 'baseline', 'verdict_swapped': 'baseline'}, {'id': 144, 'verdict_normal': 'baseline', 'verdict_swapped': 'baseline'}, {'id': 1297, 'verdict_normal': 'baseline', 'verdict_swapped': 'baseline'}, {'id': 844, 'verdict_normal': 'baseline', 'verdict_swapped': 'baseline'}, {'id': 1120, 'verdict_normal': 'baseline', 'verdict_swapped': 'baseline'}, {'id': 961, 'verdict_normal': 'baseline', 'verdict_swapped': 'baseline'}, {'id': 950, 'verdict_normal': 'baseline', 'verdict_swapped': 'baseline'}, {'id': 751, 'v

## Fase 5 -- Metricas clasicas y metrica de dominio juridico

No requieren GPU (BERTScore descarga un modelo aparte, chico, en CPU/GPU
segun disponibilidad).


In [10]:
from tools.evaluation import pipeline

eval_rows_baseline = pipeline.build_eval_rows(baseline_results, judge_baseline)
eval_rows_finetuned = pipeline.build_eval_rows(finetuned_results, judge_finetuned)

pipeline.fill_bertscore(eval_rows_baseline)
pipeline.fill_bertscore(eval_rows_finetuned)

all_rows = eval_rows_baseline + eval_rows_finetuned
print(f"Total de filas evaluadas: {len(all_rows)}")


config.json:   0%|          | 0.00/648 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/364 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/242k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/480k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Total de filas evaluadas: 402


## Fase 6 -- Sesgos: length bias y self-preference bias


In [11]:
length_bias_baseline = bias.length_bias_correlation(
    [r.judge_composite for r in eval_rows_baseline if r.judge_composite is not None],
    [len(r.generated) for r in eval_rows_baseline if r.judge_composite is not None],
)
length_bias_finetuned = bias.length_bias_correlation(
    [r.judge_composite for r in eval_rows_finetuned if r.judge_composite is not None],
    [len(r.generated) for r in eval_rows_finetuned if r.judge_composite is not None],
)

self_pref_report = bias.self_preference_gap(
    judge_baseline=[r.judge_composite for r in eval_rows_baseline if r.judge_composite is not None],
    judge_finetuned=[r.judge_composite for r in eval_rows_finetuned if r.judge_composite is not None],
    sim_baseline=[r.similarity_pct for r in eval_rows_baseline if r.similarity_pct is not None],
    sim_finetuned=[r.similarity_pct for r in eval_rows_finetuned if r.similarity_pct is not None],
)

bias_summary = {
    "length_bias_pearson_r_baseline": length_bias_baseline["pearson_r"],
    "length_bias_pearson_r_fine_tuned": length_bias_finetuned["pearson_r"],
    "self_preference_judge_gap": self_pref_report.judge_gap,
    "self_preference_similarity_gap": self_pref_report.similarity_gap,
    "self_preference_divergence": self_pref_report.divergence,
    "self_preference_flagged": self_pref_report.flagged,
    "position_bias_flip_rate_pct": position_bias_report.flip_rate_pct,
    "position_bias_n_pairs": position_bias_report.n_pairs,
}
bias_summary


{'length_bias_pearson_r_baseline': -0.411,
 'length_bias_pearson_r_fine_tuned': 0.042,
 'self_preference_judge_gap': 0.063,
 'self_preference_similarity_gap': 0.151,
 'self_preference_divergence': -0.088,
 'self_preference_flagged': False,
 'position_bias_flip_rate_pct': 0.0,
 'position_bias_n_pairs': 30}

## Fase 7 -- Scorecard y persistencia en Drive

Cada corrida se guarda en una carpeta con timestamp propio (no se sobreescribe
la corrida anterior mientras se itera).


In [12]:
import json as _json
import subprocess
from datetime import datetime, timezone
from pathlib import Path

from tools.evaluation import scorecard

manifest = pipeline.build_manifest(n_val=len(val_records), hardware=subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True,
).stdout.strip())

run_id = datetime.now(timezone.utc).strftime("%Y-%m-%d_%H%M%S")
drive_run_dir = Path(f"{config.DRIVE_EVAL_OUTPUT_ROOT}/{run_id}")
drive_run_dir.mkdir(parents=True, exist_ok=True)

summaries = scorecard.summarize_by_label(all_rows)
category_summaries = {
    "baseline": scorecard.summarize_by_category(all_rows, "baseline"),
    "fine_tuned": scorecard.summarize_by_category(all_rows, "fine_tuned"),
}
narrative = scorecard.build_narrative(summaries, bias_summary, n_val=len(val_records))

scorecard.export_markdown(
    drive_run_dir / "scorecard.md", summaries, category_summaries, narrative,
    bias_summary, manifest.__dict__,
)
scorecard.export_csv(drive_run_dir / "metricas_por_registro.csv", all_rows)

(drive_run_dir / "run_manifest.json").write_text(
    _json.dumps(manifest.__dict__, indent=2, ensure_ascii=False), encoding="utf-8"
)
(drive_run_dir / "resultados_baseline.jsonl").write_text(
    "\n".join(_json.dumps(r.__dict__, ensure_ascii=False) for r in baseline_results),
    encoding="utf-8",
)
(drive_run_dir / "resultados_finetuned.jsonl").write_text(
    "\n".join(_json.dumps(r.__dict__, ensure_ascii=False) for r in finetuned_results),
    encoding="utf-8",
)
(drive_run_dir / "position_bias_probe.json").write_text(
    _json.dumps(position_bias_report.__dict__, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print(f"Resultados guardados en: {drive_run_dir}")
print((drive_run_dir / "scorecard.md").read_text(encoding="utf-8"))


Resultados guardados en: /content/drive/MyDrive/Colab Notebooks/Amparo/evaluacion/2026-09-04_213446
# Scorecard M2 — Evaluación del Modelo

Generado: 2026-09-04T21:34:46.902269+00:00 · commit `06a8dab656023699d061ff17382c12cfbfbde2e6` · seed 42 · hardware: NVIDIA L4, 23034 MiB

## Resumen por modelo

| Modelo | N | Exact Match | F1 | BLEU | ROUGE-L | BERTScore | Similitud (%) | Juez (1-5) | Cumplimiento citas (%) | Latencia (s) |
|---|---|---|---|---|---|---|---|---|---|---|
| baseline | 201 | 0.0% | 0.177 | 1.691 | 0.122 | 69.742 | 3.411 | 4.024 | 73.6% | 12.091 |
| fine_tuned | 201 | 0.0% | 0.318 | 8.638 | 0.246 | 78.781 | 18.54 | 4.277 | 100.0% | 4.251 |

## Resumen por categoría

### baseline

| Categoría | N | Similitud (%) | Juez (1-5) | Cumplimiento citas (%) |
|---|---|---|---|---|
| Acceso a informacion publica | 8 | 2.65 | 4.125 | 37.5% |
| Accidentes de transito | 9 | 3.567 | 4.194 | 88.9% |
| Arriendo | 9 | 2.978 | 3.694 | 55.6% |
| Comparendos de transito | 9 | 3.167 | 3.9

## (Opcional) Explorar resultados en pandas


In [13]:
import pandas as pd

df = pd.DataFrame([r.__dict__ for r in all_rows])
df.groupby("label")[["similarity_pct", "judge_composite", "citation_count"]].mean()


,similarity_pct,judge_composite,citation_count
label,,,
baseline,3.411443,4.024112,0.368159
fine_tuned,18.539801,4.277363,0.000000
